## Installing the libraries

In [60]:
!pip install -q langchain langchain-community langchain-google-genai faiss-cpu pypdf pandas langchain-text-splitters sentence-transformers

IMPORTING THE LIBRARIES AND API KEY

In [61]:
import os
import pandas as pd

from google.colab import userdata

In [62]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from pypdf import PdfReader


## Loading Api key

In [124]:
from google.colab import userdata
GEMINIKEY = userdata.get('GEMINIKEY')

##Select three PDF documents


In [96]:
pdf_files = ["/content/bain_report_global-private-equity-report-2024.pdf",
            "/content/economics-of-private-equity.pdf",
            "/content/mckinsey-global-private-markets-review-2024.pdf"]

In [97]:
print("Total PDF files selected:", len(pdf_files))
for file in pdf_files:
    print(file)

Total PDF files selected: 3
/content/bain_report_global-private-equity-report-2024.pdf
/content/economics-of-private-equity.pdf
/content/mckinsey-global-private-markets-review-2024.pdf


##CHECKING PDF PAGE COUNT

In [66]:
for pdf in pdf_files:
    reader = PdfReader(pdf)
    total_pages = len(reader.pages)
    print("PDF Name:", pdf.split("/")[-1])
    print("Total Pages:", total_pages)
    print("-" * 40)

PDF Name: bain_report_global-private-equity-report-2024.pdf
Total Pages: 60
----------------------------------------
PDF Name: economics-of-private-equity.pdf
Total Pages: 61
----------------------------------------
PDF Name: mckinsey-global-private-markets-review-2024.pdf
Total Pages: 66
----------------------------------------


##Loading PDF Documents

In [67]:
all_pages = []

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    pages = loader.load()

    for page in pages:
        page.metadata["source_file"] = pdf.split("/")[-1]

    all_pages.extend(pages)

    print(pdf.split("/")[-1], "loaded")
    print("Total pages loaded:", len(all_pages))

bain_report_global-private-equity-report-2024.pdf loaded
Total pages loaded: 60
economics-of-private-equity.pdf loaded
Total pages loaded: 121
mckinsey-global-private-markets-review-2024.pdf loaded
Total pages loaded: 187


##Checking Sample Page and Metadata

In [68]:
print("Sample Metadata:")
print(all_pages[0].metadata)

print("\nSample Page Content:")
print(all_pages[0].page_content[:700])

Sample Metadata:
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.2 (Windows)', 'creationdate': '2024-03-07T11:47:59-05:00', 'moddate': '2024-03-07T16:07:52-05:00', 'trapped': '/False', 'source': '/content/bain_report_global-private-equity-report-2024.pdf', 'total_pages': 60, 'page': 0, 'page_label': '1', 'source_file': 'bain_report_global-private-equity-report-2024.pdf'}

Sample Page Content:
GLOBAL PRIVATE EQUITY REPORT 2024


## Chunking

In [69]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150)

chunks = text_splitter.split_documents(all_pages)

print("Total chunks created:", len(chunks))

Total chunks created: 604


In [70]:
first_chunk = chunks[0]

print(first_chunk.page_content[:500])
print(first_chunk.metadata)

GLOBAL PRIVATE EQUITY REPORT 2024
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.2 (Windows)', 'creationdate': '2024-03-07T11:47:59-05:00', 'moddate': '2024-03-07T16:07:52-05:00', 'trapped': '/False', 'source': '/content/bain_report_global-private-equity-report-2024.pdf', 'total_pages': 60, 'page': 0, 'page_label': '1', 'source_file': 'bain_report_global-private-equity-report-2024.pdf'}


##Adding metadata

In [71]:
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_number"] = i + 1
    chunk.metadata["short_preview"] = chunk.page_content[:100]

In [72]:
print("Metadata added successfully.")

Metadata added successfully.


In [73]:
# first chunk metadata
print("Chunk number:", chunks[0].metadata["chunk_number"])
print("Source file:", chunks[0].metadata.get("source_file"))
print("Short preview:", chunks[0].metadata["short_preview"])

Chunk number: 1
Source file: bain_report_global-private-equity-report-2024.pdf
Short preview: GLOBAL PRIVATE EQUITY REPORT 2024


##Creating and saving FAISS vector database

In [74]:
print("Total chunks:", len(chunks))
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")

# Creating the vector database
vector_db = FAISS.from_documents(documents=chunks,embedding=embedding_model)

# Saving the vector
vector_db.save_local("private_equity_vector_database")

print("Vector database created and saved successfully.")

Total chunks: 604


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database created and saved successfully.


SIMILARITY CHECK

In [75]:
question = "What is dry powder in private equity?"

results = vector_db.similarity_search(question, k=4)

for result in results:
    print(result.page_content[:500])
    print(result.metadata)
    print("-" * 50)

31 percent as of June 30, 2023.
PE dry powder grew to a record $2.2 trillion through 
the first half of 2023, driven by a dramatic slowdown 
in dealmaking while fundraising continued, albeit at 
a slower pace. Dry-powder inventory (the amount of 
capital available to GPs expressed as a multiple of 
annual deployment) increased from 1.1 years in 2022 
to 1.6 years in 2023 but remains within the metric’s 
normal historical range (Exhibit 15). 
Exhibit 15
Global inventories of PE dry powder continu
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.3 (Macintosh)', 'creationdate': '2024-04-12T10:38:17-05:00', 'moddate': '2024-04-12T10:38:32-05:00', 'trapped': '/False', 'source': '/content/mckinsey-global-private-markets-review-2024.pdf', 'total_pages': 66, 'page': 24, 'page_label': '23', 'source_file': 'mckinsey-global-private-markets-review-2024.pdf', 'chunk_number': 485, 'short_preview': '31 percent as of June 30, 2023.\nPE dry powder grew to a record $2.2 trillion thro



##Connecting with Gemini LLM

In [125]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINIKEY,
    temperature=0.2)

print("Gemini connected successfully.")

Gemini connected successfully.


#RETRIEVING RELEVANT CHUNK

In [77]:
def retrieve_context(question):
    results = vector_db.similarity_search(question, k=3)

    context = ""
    sources = []

    for doc in results:
        context = context + doc.page_content[:1200]
        sources.append(doc.metadata)

    return context, sources

##Creating the prompting strategy

one shot prompt

In [78]:
one_shot_template = """
You are answering questions using private equity documents.
Example:
Question: What is dry powder?
Answer: Dry powder means capital that investors have committed but private equity firms have not yet invested.

Use only the context below to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

In [79]:
one_shot_prompt = PromptTemplate(input_variables=["context", "question"],
    template=one_shot_template)

few shot prompt

In [80]:
few_shot_template = """
You are answering questions using private equity documents.

Example 1:
Question: What is a GP?
Answer: A GP, or General Partner, manages the private equity fund.

Example 2:
Question: What is an LP?
Answer: An LP, or Limited Partner, provides capital to the private equity fund.

Example 3:
Question: What is carried interest?
Answer: Carried interest is the share of profits earned by private equity fund managers.

Use only the context below to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""
few_shot_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=few_shot_template)

step by step prompt

In [81]:
step_by_step_template = """
Use the context below to answer the question.

Follow this structure:
1. What the question is asking.
2. What the document evidence says.
3. Final answer in simple words.

Context:
{context}

Question:
{question}

Answer:
"""

step_by_step_prompt = PromptTemplate(input_variables=["context", "question"],
    template=step_by_step_template)

Refining prompt

In [82]:
sample_question = "What is dry powder and why is it important in private equity?"

context, sources = retrieve_context(sample_question)

# First prompt - simple prompt
basic_prompt = f"""
Use the context below to answer the question.

Context:
{context}

Question:
{sample_question}

Answer:
"""

In [83]:
# Second prompt - improved prompt with source focus
refined_prompt_1 = f"""
Use only the context below to answer the question.
Do not use outside knowledge.

Context:
{context}

Question:
{sample_question}

Answer in simple words:
"""

In [84]:
# Third prompt - final improved prompt with structure
refined_prompt_2 = f"""
Use only the context below to answer the question.

Follow this structure:
1. What the question is asking
2. What the document evidence says
3. Final answer in simple words

Context:
{context}

Question:
{sample_question}

Answer:
"""


In [85]:
print("Basic prompt created")
print("First refined prompt created")
print("Final refined prompt created")

Basic prompt created
First refined prompt created
Final refined prompt created


In [108]:
basic_response = llm.invoke(basic_prompt)
refined_response_1 = llm.invoke(refined_prompt_1)
refined_response_2 = llm.invoke(refined_prompt_2)

print("BASIC PROMPT ANSWER:")
print(basic_response.content)
print("-" * 80)

print("REFINED PROMPT 1 ANSWER:")
print(refined_response_1.content)
print("-" * 80)

print("REFINED PROMPT 2 ANSWER:")
print(refined_response_2.content)
print("-" * 80)

BASIC PROMPT ANSWER:
In private equity, **dry powder** refers to the amount of capital that has been committed by investors to private equity funds but has not yet been invested (deployed) by the General Partners (GPs).

It is important because it represents the future investment capacity of private equity firms. The context highlights its significance in several ways:

*   **Indicator of market dynamics:** Its growth to record levels (e.g., $2.2 trillion or $3.7 trillion as of June 30, 2023) is attributed to a dramatic slowdown in dealmaking while fundraising continued, albeit at a slower pace. This indicates market conditions where capital is being raised faster than it's being deployed.
*   **Measure of investment capacity:** "Dry-powder inventory" (the amount of capital available to GPs expressed as a multiple of annual deployment, such as 1.6 years in 2023) is a key metric. It indicates how many years GPs could continue deploying capital at current rates, reflecting the balance be

##Creating function to ask from gemini

In [110]:
def ask_question(question, strategy):
    context, sources = retrieve_context(question)

    if strategy == "one_shot":
        final_prompt = one_shot_prompt.format(
            context=context,
            question=question)

    elif strategy == "few_shot":
        final_prompt = few_shot_prompt.format(
            context=context,
            question=question)
    else:
        final_prompt = step_by_step_prompt.format(
            context=context,
            question=question)

    response = llm.invoke(final_prompt)

    result = {"question": question,
        "strategy": strategy,
        "answer": response.content,
        "model": llm.model,
        "input_tokens_estimate": len(final_prompt.split()),
        "output_tokens_estimate": len(response.content.split()),
        "sources": sources}

    return result

##Preparing Five Main RAG Questions

In [111]:
questions = [{"question": "What were the main reasons for the slowdown in private equity deals in 2023?",
        "strategy": "one_shot"},
    {"question": "What is dry powder and why is it important in private equity?",
        "strategy": "few_shot"},
    {"question": "How did higher interest rates affect private equity activity?",
        "strategy": "step_by_step"},
    {"question": "How did private credit perform compared with private equity?",
        "strategy": "few_shot"},
    {"question": "What are the main ways private equity firms create value after buying a company?",
        "strategy": "step_by_step"}]

##Running Five Main Questions

In [112]:
all_results = []

for item in questions:
    try:
        result = ask_question(
            question=item["question"],
            strategy=item["strategy"])

        all_results.append(result)

        print("Question:", result["question"])
        print("Strategy:", result["strategy"])
        print("Model:", result["model"])
        print("Input tokens estimate:", result["input_tokens_estimate"])
        print("Output tokens estimate:", result["output_tokens_estimate"])

        print("Answer:")
        print(result["answer"])

        print("Sources:")
        for source in result["sources"]:
            print(source)

        print("-" * 80)

    except Exception as e:
        print("Error for question:", item["question"])
        print(e)
        print("-" * 80)

Question: What were the main reasons for the slowdown in private equity deals in 2023?
Strategy: one_shot
Model: gemini-2.5-flash
Input tokens estimate: 535
Output tokens estimate: 102
Answer:
The main reasons for the slowdown in private equity deals in 2023 were:

*   Rapidly rising interest rates, as the Fed jacked rates at the fastest pace since the 1980s, which led to a loss of confidence.
*   Exit channels largely drying up, leaving general partners (GPs) with a towering $3.2 trillion in unsold assets and stanching the flow of capital back to limited partners (LPs).
*   Slower distributions, which left LPs cash flow negative and crimped their ability to plow more capital back into private equity, causing them to pull back new allocations from all but the largest, most reliable funds.
Sources:
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.2 (Windows)', 'creationdate': '2024-03-07T11:47:59-05:00', 'moddate': '2024-03-07T16:07:52-05:00', 'trapped': '/False', 's

In [113]:
print("Question:", result["question"])
print("Strategy:", result["strategy"])
print("Model:", result["model"])
print("Input tokens estimate:", result["input_tokens_estimate"])
print("Output tokens estimate:", result["output_tokens_estimate"])

print("Answer:")
print(result["answer"])
print("Sources:")
for source in result["sources"]:
    print(source)

print("-" * 80)

Question: What are the main ways private equity firms create value after buying a company?
Strategy: step_by_step
Model: gemini-2.5-flash
Input tokens estimate: 475
Output tokens estimate: 185
Answer:
1.  **What the question is asking.**
    The question asks for the primary methods private equity (PE) firms use to create value *after* they have acquired a company.

2.  **What the document evidence says.**
    The document states:
    *   Cohn, Hotchkiss, and Towery (2022) explore "profitability improvements, financial engineering, and easing financial constraints as potential value sources." They also note "moderate profitability improvement and significant sales growth" post-buyout, indicating that "unlocking growth potential is crucial."
    *   Biesinger, Bircan, and Ljungqvist (2023) conclude that PE funds create financial value through "value-add activities focused on the asset side of the balance sheet." They specifically state that "financial engineering activities on the liabi

##Creating result table

In [114]:
print("Number of results:", len(all_results))

Number of results: 5


In [115]:
results_table = []

for result in all_results:
    source_names = []

    for source in result["sources"]:
        if "source_file" in source:
            source_names.append(source["source_file"])

    source_names = list(set(source_names))

    results_table.append({
        "Question": result["question"],
        "Prompting Strategy": result["strategy"],
        "Model": result["model"],
        "Input Tokens Estimate": result["input_tokens_estimate"],
        "Output Tokens Estimate": result["output_tokens_estimate"],
        "Sources": ", ".join(source_names),
        "Answer": result["answer"]})

df_results = pd.DataFrame(results_table)

df_results

,Question,Prompting Strategy,Model,Input Tokens Estimate,Output Tokens Estimate,Sources,Answer
0,What were the main reasons for the slowdown in...,one_shot,gemini-2.5-flash,535,102,bain_report_global-private-equity-report-2024.pdf,The main reasons for the slowdown in private e...
1,What is dry powder and why is it important in ...,few_shot,gemini-2.5-flash,424,50,mckinsey-global-private-markets-review-2024.pdf,Dry powder is the amount of capital committed ...
2,How did higher interest rates affect private e...,step_by_step,gemini-2.5-flash,510,143,bain_report_global-private-equity-report-2024....,1. **What the question is asking.**\n The ...
3,How did private credit perform compared with p...,few_shot,gemini-2.5-flash,517,48,economics-of-private-equity.pdf,The provided context describes private debt (P...
4,What are the main ways private equity firms cr...,step_by_step,gemini-2.5-flash,475,185,economics-of-private-equity.pdf,1. **What the question is asking.**\n The ...


##saving result

In [116]:
df_results.to_csv("part_2_rag_results.csv", index=False)

print("Results saved as part_2_rag_results.csv")

Results saved as part_2_rag_results.csv


##Conversations with the Dataset

In [117]:
conversation_1_q1 = ask_question("What happened to private equity deal activity in 2023?", "one_shot")
conversation_1_q2 = ask_question("Why did higher interest rates affect private equity deals?", "step_by_step")

print(conversation_1_q1["answer"])
print(conversation_1_q2["answer"])

Private equity deal activity experienced sharp declines in 2023. Rapidly rising interest rates led to sharp declines in dealmaking. Buyout investment value dropped to $438 billion, a 37% decrease from 2022. The falloff that started in the second half of 2022 bled over into 2023.
1.  **What the question is asking.**
    The question asks why higher interest rates impacted private equity deals.

2.  **What the document evidence says.**
    The document states that higher interest rates have made acquisitions harder by raising the cost of capital. They have also complicated dealmaking by putting downward pressure on price multiples, as multiples of cash flow go down when rates go up. Investors have responded by using less debt and more equity, leading to a drop in debt multiples. Furthermore, the cost of financing has doubled, making it significantly harder for companies to service existing debt and fund new acquisition strategies. This also makes multiple arbitrage harder to capture beca

In [118]:
conversation_2_q1 = ask_question("How can generative AI help private equity firms?", "few_shot")
conversation_2_q2 = ask_question("How can generative AI support due diligence?", "step_by_step")

print(conversation_2_q1["answer"])
print(conversation_2_q2["answer"])

Generative AI can help private equity firms by transforming portfolio companies, sharpening due diligence, and making investment professionals smarter. Within the portfolio, firms use it to gain insight into potential impacts and opportunities across companies and industries. In due diligence, leading teams are developing protocols to assess generative AI threats and opportunities, and firms are using AI tools to speed up and sharpen the process. It can also be deployed to improve processes and make people more efficient by linking to pragmatic business objectives.
Here's how generative AI can support due diligence:

1.  **What the question is asking.**
    The question asks how generative AI can assist or improve the process of due diligence.

2.  **What the document evidence says.**
    The document states that generative AI can be used for "routine as legal or commercial diligence." It also mentions that "Firms are also using AI tools to speed up and sharpen the underwriting process

In [119]:
conversation_3_q1 = ask_question("How do private equity firms create value after buying a company?", "step_by_step")
conversation_3_q2 = ask_question("What challenges can affect private equity returns?", "few_shot")

print(conversation_3_q1["answer"])
print(conversation_3_q2["answer"])

1.  **What the question is asking.**
    The question asks to identify the methods or strategies private equity (PE) firms use to create value in a company after they have acquired it.

2.  **What the document evidence says.**
    The document states that "Post-buyout, there is moderate profitability improvement and significant sales growth, indicating that unlocking growth potential is crucial." It also references a study by Cohn, Hotchkiss, and Towery (2022) which explores "profitability improvements, financial engineering, and easing financial constraints as potential value sources" in private firm buyouts. The text emphasizes that true value creation is the part of the valuation increase that would not have happened "but for" the PE fund's investment, implying active involvement beyond just picking companies that would improve anyway.

3.  **Final answer in simple words.**
    After buying a company, private equity firms create value by improving its profitability, achieving signif

## Hallucination and Error Checking

In [120]:
hallucination_questions = [
    "Which private equity firm will definitely perform best in 2025?",
    "What is the exact share price forecast for Bain & Company?",
    "What does the dataset say about cryptocurrency investments in private equity?"]

hallucination_results = []

for q in hallucination_questions:
    result = ask_question(q, "step_by_step")
    hallucination_results.append(result)

    print("Question:", result["question"])
    print("Answer:")
    print(result["answer"])
    print("Sources:")
    print(result["sources"])
    print("-" * 80)

Question: Which private equity firm will definitely perform best in 2025?
Answer:
1.  **What the question is asking:** The question asks to identify a specific private equity firm that will *definitely* perform best in 2025.

2.  **What the document evidence says:** The provided text discusses the overall state of the private equity industry, its resilience, investment trends (a drop in buyout investment value in 2023), and returns (10-year IRR trending slightly downward but still beating public markets). It mentions challenges like high interest rates affecting confidence. However, the document *does not mention any specific private equity firms by name*, nor does it make any predictions about the future performance of individual firms.

3.  **Final answer in simple words:** The document does not provide information about specific private equity firms or predict which one will perform best in 2025.
Sources:
[{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.2 (Windo

##Saving All Final Answers

In [121]:
main_results_df = pd.DataFrame(df_results)
main_results_df.to_csv("part2_main_questions_results.csv", index=False)

print("Main question results saved.")

Main question results saved.


In [122]:
save_folder = "/content/drive/MyDrive/BA_Part2_Saved_Work"
os.makedirs(save_folder, exist_ok=True)

conversation_results = [conversation_1_q1,conversation_1_q2,conversation_2_q1,conversation_2_q2,conversation_3_q1,
    conversation_3_q2]

conversation_table = pd.DataFrame(conversation_results)

conversation_table.to_csv(save_folder + "/conversation_results.csv", index=False)

print("Conversation results saved")

Conversation results saved


SAVING THE REFINED PROMPT

In [123]:

text = """
BASIC PROMPT ANSWER:
""" + basic_response.content + """

REFINED PROMPT 1 ANSWER:
""" + refined_response_1.content + """

REFINED PROMPT 2 ANSWER:
""" + refined_response_2.content

file = open("prompt_refinement_answers.txt", "w")
file.write(text)
file.close()

print("Saved successfully")

Saved successfully
